In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from diffusers.optimization import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm
import os
import pandas as pd
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

# --- デバイスの設定 ---
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
# --- ハイパーパラメータ ---
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
NUM_EPOCHS = 150
LAMBDA_L1 = 100.0 # Pix2PixのL1損失の重み（標準値）

IMAGE_DIR = '/home/masaki/mystudy/my_yale_data/yale_t1post_raw'
CSV_FILE = '/home/masaki/mystudy/my_yale_data/final_labeled_data.csv'

# レビュー対応: Pix2Pix用の保存フォルダ
OUTPUT_DIR = "/home/masaki/mystudy/trained_models_masked_loss_+Pix2Pix"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

Output directory ready: /home/masaki/mystudy/trained_models_masked_loss_+Pix2Pix


In [3]:
# CSVを読み込んで、ユニークな患者IDのリストを取得
full_df = pd.read_csv(CSV_FILE)
patient_ids = full_df['PatientID'].unique()

# ステップ1: 学習用(70%)と、残り(30%)に分割
train_patient_ids, temp_patient_ids = train_test_split(
    patient_ids, test_size=0.3, random_state=42
)

# ステップ2: 残り(30%)を、検証用(15%)とテスト用(15%)に分割
val_patient_ids, test_patient_ids = train_test_split(
    temp_patient_ids, test_size=0.5, random_state=42
)

print(f"Total patients: {len(patient_ids)}")
print(f"Training patients: {len(train_patient_ids)}")
print(f"Validation patients: {len(val_patient_ids)}")
print(f"Test patients: {len(test_patient_ids)}")

Total patients: 1362
Training patients: 953
Validation patients: 204
Test patients: 205


In [4]:
class BrainMetsDataset(Dataset):
    def __init__(self, csv_file, image_root_dir, patient_ids, mode, image_size=256, latent_size=32):
        self.data = pd.read_csv(csv_file)
        self.data = self.data[self.data['PatientID'].isin(patient_ids)]
        
        self.image_root_dir = image_root_dir
        self.seg_dir = '/home/masaki/mystudy/my_yale_data/yale_brain_mets_segmentations'
        self.mode = mode
        self.image_size = image_size
        self.latent_size = latent_size
        self.pairs = self._create_pairs()
        
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((self.image_size, self.image_size), antialias=True),
            transforms.Normalize([0.5], [0.5])
        ])
        self.mask_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((self.latent_size, self.latent_size), antialias=False)
        ])

    def _create_pairs(self):
        pairs = []
        for _, group in self.data.groupby('PatientID'):
            baseline_scans = group[group['FinalPattern'] == 'Baseline']
            future_scans = group[group['FinalPattern'] != 'Baseline']
            if baseline_scans.empty or future_scans.empty: continue
            for _, baseline_row in baseline_scans.iterrows():
                for _, future_row in future_scans.iterrows():
                    pairs.append({
                        "baseline_file": baseline_row['FileName'],
                        "future_file": future_row['FileName'],
                        "condition": future_row['FinalPattern']
                    })
        print(f"Found {len(pairs)} pairs for '{self.mode}' set.")
        return pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        try:
            baseline_path = self._get_full_path(self.image_root_dir, pair['baseline_file'])
            future_path = self._get_full_path(self.image_root_dir, pair['future_file'])
            mask_path = self._get_full_path(self.seg_dir, pair['future_file'], is_mask=True)

            baseline_image = self._load_and_process_image(baseline_path)
            future_image = self._load_and_process_image(future_path)
            future_mask = self._load_and_process_mask(mask_path)

            return {
                "baseline_image": baseline_image,
                "future_image": future_image,
                "condition_label": pair['condition'],
                "future_mask": future_mask,
                "baseline_filename": pair['baseline_file']
            }
        except FileNotFoundError as e:
            return self.__getitem__((idx + 1) % len(self))
            
    def _get_full_path(self, directory, filename, is_mask=False):
        name_without_ext = filename.replace('.nii.gz', '')
        suffix = "_0000.nii.gz" if not is_mask else ".nii.gz"
        return os.path.join(directory, f"{name_without_ext}{suffix}")

    def _load_and_process_image(self, path):
        nii = nib.load(path)
        data = nii.get_fdata().astype('float32')
        slice_img = data[:, :, data.shape[2] // 2]
        if slice_img.max() > slice_img.min():
            slice_img = (slice_img - slice_img.min()) / (slice_img.max() - slice_img.min())
        image = self.transform(slice_img).repeat(3, 1, 1) # 1ch -> 3chに拡張
        return image

    def _load_and_process_mask(self, path):
        nii = nib.load(path)
        data = nii.get_fdata().astype('float32')
        slice_img = data[:, :, data.shape[2] // 2]
        return self.mask_transform(slice_img)

# ローダーの作成
train_dataset = BrainMetsDataset(csv_file=CSV_FILE, image_root_dir=IMAGE_DIR, patient_ids=train_patient_ids, mode='train')
val_dataset = BrainMetsDataset(csv_file=CSV_FILE, image_root_dir=IMAGE_DIR, patient_ids=val_patient_ids, mode='val')

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Found 5461 pairs for 'train' set.
Found 1082 pairs for 'val' set.


In [5]:
# 重みの初期化関数（GANの安定学習に必須）
def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        torch.nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm2d") != -1:
        torch.nn.init.normal_(m.weight.data, 1.0, 0.02)
        torch.nn.init.constant_(m.bias.data, 0.0)

# --- Generator (U-Net) ---
class UNetDown(nn.Module):
    def __init__(self, in_size, out_size, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_size, out_size, 4, 2, 1, bias=False)]
        if normalize: layers.append(nn.BatchNorm2d(out_size))
        layers.append(nn.LeakyReLU(0.2))
        if dropout: layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x): return self.model(x)

class UNetUp(nn.Module):
    def __init__(self, in_size, out_size, dropout=0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_size, out_size, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_size),
            nn.ReLU(inplace=True)
        ]
        if dropout: layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x, skip_input):
        x = self.model(x)
        return torch.cat((x, skip_input), 1)

class GeneratorUNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=3): # Baseline(3) + Label(1) = 4
        super().__init__()
        self.down1 = UNetDown(in_channels, 64, normalize=False)
        self.down2 = UNetDown(64, 128)
        self.down3 = UNetDown(128, 256)
        self.down4 = UNetDown(256, 512, dropout=0.5)
        self.down5 = UNetDown(512, 512, dropout=0.5)
        self.down6 = UNetDown(512, 512, dropout=0.5)
        self.down7 = UNetDown(512, 512, dropout=0.5)
        self.down8 = UNetDown(512, 512, normalize=False, dropout=0.5)
        
        self.up1 = UNetUp(512, 512, dropout=0.5)
        self.up2 = UNetUp(1024, 512, dropout=0.5)
        self.up3 = UNetUp(1024, 512, dropout=0.5)
        self.up4 = UNetUp(1024, 512, dropout=0.5)
        self.up5 = UNetUp(1024, 256)
        self.up6 = UNetUp(512, 128)
        self.up7 = UNetUp(256, 64)
        
        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_channels, 4, 2, 1),
            nn.Tanh()
        )
    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        d8 = self.down8(d7)
        u1 = self.up1(d8, d7)
        u2 = self.up2(u1, d6)
        u3 = self.up3(u2, d5)
        u4 = self.up4(u3, d4)
        u5 = self.up5(u4, d3)
        u6 = self.up6(u5, d2)
        u7 = self.up7(u6, d1)
        return self.final(u7)

# --- Discriminator (PatchGAN) ---
class Discriminator(nn.Module):
    def __init__(self, in_channels=7): # Generator Input(4) + Target Future(3) = 7
        super().__init__()
        def discriminator_block(in_filters, out_filters, normalization=True):
            layers = [nn.Conv2d(in_filters, out_filters, 4, stride=2, padding=1)]
            if normalization: layers.append(nn.BatchNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        self.model = nn.Sequential(
            *discriminator_block(in_channels, 64, normalization=False),
            *discriminator_block(64, 128),
            *discriminator_block(128, 256),
            *discriminator_block(256, 512),
            nn.Conv2d(512, 1, 3, padding=1, bias=False) # Patch output
        )
    def forward(self, img_input, target):
        x = torch.cat((img_input, target), 1)
        return self.model(x)

# モデルのインスタンス化と初期化
generator = GeneratorUNet().to(device)
discriminator = Discriminator().to(device)
generator.apply(weights_init_normal)
discriminator.apply(weights_init_normal)
print("Pix2Pix Models initialized.")

Pix2Pix Models initialized.


In [6]:
# 損失関数と最適化
criterion_GAN = torch.nn.BCEWithLogitsLoss().to(device) # パッチ用
criterion_pixelwise = torch.nn.L1Loss().to(device)      # ピクセル誤差

optimizer_G = torch.optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))

train_losses_g = []
train_losses_d = []
val_losses_l1 = []
best_val_l1 = float('inf')

print("Starting Pix2Pix Training...")
for epoch in range(NUM_EPOCHS):
    generator.train()
    discriminator.train()
    running_loss_g = 0.0
    running_loss_d = 0.0
    
    # === 学習 ===
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]"):
        real_base = batch["baseline_image"].to(device)
        real_future = batch["future_image"].to(device)
        
        # ラベル(条件)を画像チャネルに変換
        b, _, h, w = real_base.shape
        label_imgs = []
        for label in batch["condition_label"]:
            val = 1.0 if label == "Progression" else -1.0
            label_imgs.append(torch.full((1, h, w), val, device=device))
        label_img_tensor = torch.stack(label_imgs)
        
        # 入力の作成：Baseline(3ch) + Label(1ch) = 4ch
        input_gen = torch.cat([real_base, label_img_tensor], 1)

        # ------------------
        #  Train Generator
        # ------------------
        optimizer_G.zero_grad()
        fake_future = generator(input_gen)
        
        pred_fake = discriminator(input_gen, fake_future)
        loss_GAN = criterion_GAN(pred_fake, torch.ones_like(pred_fake))
        loss_pixel = criterion_pixelwise(fake_future, real_future)
        
        loss_G = loss_GAN + LAMBDA_L1 * loss_pixel # トータルGenerator Loss
        loss_G.backward()
        optimizer_G.step()
        
        # ---------------------
        #  Train Discriminator
        # ---------------------
        optimizer_D.zero_grad()
        pred_real = discriminator(input_gen, real_future)
        loss_real = criterion_GAN(pred_real, torch.ones_like(pred_real))
        
        pred_fake = discriminator(input_gen, fake_future.detach())
        loss_fake = criterion_GAN(pred_fake, torch.zeros_like(pred_fake))
        
        loss_D = 0.5 * (loss_real + loss_fake)
        loss_D.backward()
        optimizer_D.step()

        running_loss_g += loss_G.item()
        running_loss_d += loss_D.item()

    # === 検証 ===
    generator.eval()
    val_loss_l1 = 0.0
    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
            real_base = batch["baseline_image"].to(device)
            real_future = batch["future_image"].to(device)
            
            b, _, h, w = real_base.shape
            label_imgs = []
            for label in batch["condition_label"]:
                val = 1.0 if label == "Progression" else -1.0
                label_imgs.append(torch.full((1, h, w), val, device=device))
            label_img_tensor = torch.stack(label_imgs)
            
            input_gen = torch.cat([real_base, label_img_tensor], 1)
            fake_future = generator(input_gen)
            
            loss_pixel = criterion_pixelwise(fake_future, real_future)
            val_loss_l1 += loss_pixel.item()

    avg_train_loss_g = running_loss_g / len(train_dataloader)
    avg_train_loss_d = running_loss_d / len(train_dataloader)
    avg_val_loss_l1 = val_loss_l1 / len(val_dataloader)
    
    train_losses_g.append(avg_train_loss_g)
    train_losses_d.append(avg_train_loss_d)
    val_losses_l1.append(avg_val_loss_l1)
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Train G: {avg_train_loss_g:.4f}, Train D: {avg_train_loss_d:.4f}, Val L1: {avg_val_loss_l1:.4f}")
    
    # ベストモデルの保存
    if avg_val_loss_l1 < best_val_l1:
        best_val_l1 = avg_val_loss_l1
        save_path = os.path.join(OUTPUT_DIR, "trained_models_masked_loss_+Pix2Pix.pth")
        torch.save(generator.state_dict(), save_path)
        print(f"🌟 Best model saved to {save_path} (Val L1: {best_val_l1:.4f})")

print("Training Complete!")

Starting Pix2Pix Training...


Epoch 1/150 [Train]:   0%|          | 0/2730 [00:00<?, ?it/s]

Epoch 1/150 [Val]:   0%|          | 0/541 [00:00<?, ?it/s]

Epoch [1/150] Train G: 11.3189, Train D: 0.4770, Val L1: 0.0905
🌟 Best model saved to /home/masaki/mystudy/trained_models_masked_loss_+Pix2Pix/trained_models_masked_loss_+Pix2Pix.pth (Val L1: 0.0905)


Epoch 2/150 [Train]:   0%|          | 0/2730 [00:00<?, ?it/s]

Epoch 2/150 [Val]:   0%|          | 0/541 [00:00<?, ?it/s]

... (441 outputs left out)

Epoch [149/150] Train G: 12.8975, Train D: 0.1589, Val L1: 0.0978


Epoch 150/150 [Train]:   0%|          | 0/2730 [00:00<?, ?it/s]

Epoch 150/150 [Val]:   0%|          | 0/541 [00:00<?, ?it/s]

Epoch [150/150] Train G: 12.9631, Train D: 0.1576, Val L1: 0.1022
Training Complete!
